<a href="https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/pipeline_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline Audit — full test battery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/pipeline_audit.ipynb?flush_cache=true)

This notebook exists for one reason: to catch any bugs or a stale values before they reach the paper, not after. It doesn't trust the capstone notebook, the committed JSON, or its own first answer — every check here either re-derives a number from raw data by a different path than the pipeline used, or diffs a freshly-regenerated artifact against what's currently committed on GitHub.

To use it simply run top to bottom. Every section prints PASS/WARN/FAIL as it goes. The last cell fails loudly (raises) if anything hard failed — treat that as a stop sign, not a suggestion. WARN means "worth a look" (e.g. an empirical finding that could have drifted), not "something is broken."

Needs a Hugging Face READ token, same as the capstone notebook's Setup section.


## 0. Test harness

A plain list of results instead of raising on the first failed `assert` — the whole point is seeing every problem in one pass, not fixing them one at a time across five reruns.

In [ ]:
RESULTS = []

def check(name, passed, detail="", severity="fail"):
    """Record one test result. severity='fail' blocks the final summary;
    severity='warn' surfaces but doesn't block (empirical findings, not code bugs)."""
    if passed:
        status = "PASS"
    else:
        status = "WARN" if severity == "warn" else "FAIL"
    RESULTS.append({"test": name, "status": status, "detail": detail})
    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}[status]
    print(f"{icon} {name}" + (f" — {detail}" if detail else ""))
    return passed

def close(a, b, tol):
    return abs(a - b) <= tol


## 1. Setup — fresh clone, snapshot what's committed, then rebuild everything

1. Force a fresh clone (delete any stale local copy first) to test what's currently in the GitHub repo.
2. Snapshot `work/outputs/` immediately after cloning, before running anything. That snapshot is "committed" for every comparison below.
3. Rebuild everything so `work/outputs/` now holds a fresh run. That's "live" for every comparison below.


In [ ]:
import os, shutil

REPO_URL = "https://github.com/tkg-create/FlyRank-ML-Track.git"
REPO_DIR = "FlyRank-ML-Track"

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone {REPO_URL}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
assert os.path.isfile("work/scripts/01_load_and_score.py"), "01_load_and_score.py not found — did the clone work?"


Cloning into 'FlyRank-ML-Track'...
remote: Enumerating objects: 499, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 499 (delta 104), reused 43 (delta 32), pack-reused 336 (from 2)
Receiving objects: 100% (499/499), 2.14 MiB | 9.05 MiB/s, done.
Resolving deltas: 100% (290/290), done.
Working dir: /content/FlyRank-ML-Track


In [ ]:
# Snapshot committed outputs before regenerating anything.
COMMITTED_DIR = "/content/committed_outputs_snapshot"
if os.path.isdir(COMMITTED_DIR):
    shutil.rmtree(COMMITTED_DIR)
shutil.copytree("work/outputs", COMMITTED_DIR)
print("Snapshotted committed work/outputs/ to", COMMITTED_DIR)
print(sorted(os.listdir(COMMITTED_DIR)))


Snapshotted committed work/outputs/ to /content/committed_outputs_snapshot
['capstone_precision_at_k.json', 'charts', 'feature_importance.json', 'fold_representation_check.json', 'queue_diagnostics.json', 'sentinel_fill_check.json', 'w07_metrics.json', 'w07_report.md']


In [ ]:
import subprocess, sys
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face READ token: ")

def run_step(script):
    print(f"\n{'='*70}\n▶ {script}\n{'='*70}", flush=True)
    process = subprocess.Popen(
        [sys.executable, "-u", f"work/scripts/{script}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    assert process.returncode == 0, f"{script} failed with exit code {process.returncode}"

# 01 + 02 build the live model_df / queue_df and overwrite work/outputs/ with a fresh run.
run_step("01_load_and_score.py")
run_step("02_build_queue.py")
# 03 + 04 are audit scripts in their own right, they're run as well so their JSON is fresh.
run_step("03_validate_deployed_score.py")
run_step("04_check_fold_representation.py")


Paste your Hugging Face READ token: ··········

▶ 01_load_and_score.py
Loading and building features from the warehouse...
  [1/6] Querying label (impressions first half vs second half)...
        -> 151,981 rows
  [2/6] Querying full-month position/click signal (baseline rule)...
        -> 175,304 rows
  [3/6] Querying first-half-only features (model training data)...
        -> 150,675 rows
  [4/6] Querying leakage-safe position trend (week 1 vs week 2)...
        -> 119,176 rows
  [5/6] Querying client map (grouping key only, never a feature)...
        -> 331,437 rows
  [6/6] Merging into model_df...
model_df: (150675, 16), base rate: 0.438
Running 5-fold GroupKFold OOF scoring (random_state=42)...
  Fold 1/5: fitting on 120,583 rows, scoring 30,092...
  Fold 1/5: done
  Fold 2/5: fitting on 120,589 rows, scoring 30,086...
  Fold 2/5: done
  Fold 3/5: fitting on 120,351 rows, scoring 30,324...
  Fold 3/5: done
  Fold 4/5: fitting on 120,589 rows, scoring 30,086...
  Fold 4/5: done

In [ ]:
import json
import numpy as np
import pandas as pd
import duckdb
sys.path.insert(0, "work/scripts")
from w07_pipeline_utils import (
    FEATURE_COLS, HALF_SPLIT_DATE, WEEK1_END_DATE, HF_MONTH_PATH, SCORED_MONTH,
    N_FOLDS, RANDOM_STATE, HIGH_SCORE_PERCENTILE, LARGE_SWING_PERCENTILE,
    LOW_DATA_PERCENTILE, BOUNDARY_MARGIN_PCT,
)

model_df = pd.read_csv("work/data/processed/w07_scored_population.csv")
queue_df = pd.read_csv("work/data/processed/w07_ranked_queue.csv")

def load_committed(name):
    with open(f"{COMMITTED_DIR}/{name}") as f:
        return json.load(f)

def load_live(name):
    with open(f"work/outputs/{name}") as f:
        return json.load(f)

print(f"model_df: {model_df.shape}")
print(f"queue_df: {queue_df.shape}")


model_df: (150675, 19)
queue_df: (150675, 24)


In [ ]:
import pandas as pd
import numpy as np

try:
    model_df
except NameError:
    model_df = pd.read_csv("work/data/processed/w07_scored_population.csv")

# First-half-only version of the zero-click flag, built from columns model_df already has (avg_position_fh, total_impressions_fh, total_clicks_fh) — no warehouse query needed.
# Eligibility is gated on first-half impressions, not full-month, so every input this flag uses is knowable by the same day-15 cutoff the model is held to.
eligible_fh = model_df["total_impressions_fh"] >= 10
zero_clicks_at_position_fh = (
    (model_df["avg_position_fh"] <= 10) & (model_df["total_clicks_fh"] == 0) & eligible_fh
).astype(int)

print("Full-month zero-click rate:      ", round(model_df["zero_clicks_at_position"].mean(), 3))
print("First-half-only zero-click rate: ", round(zero_clicks_at_position_fh.mean(), 3))
print("Rows where the two versions disagree:",
      (model_df["zero_clicks_at_position"] != zero_clicks_at_position_fh).sum(), "/", len(model_df))

Full-month zero-click rate:       0.166
First-half-only zero-click rate:  0.199
Rows where the two versions disagree: 16210 / 150675


In [ ]:
KS = [20, 50, 100, 200]

def precision_at_k(labels_sorted, k):
    return float(np.asarray(labels_sorted)[:k].mean())

records = []
for fold_num in sorted(model_df["fold_id"].unique()):
    fold = model_df[model_df["fold_id"] == fold_num].copy()
    fold["baseline_score_fh"] = zero_clicks_at_position_fh.loc[fold.index] * 2 + fold["position_worsened"]

    order_full = fold.sort_values(["baseline_score", "total_impressions_full"], ascending=[False, False]).index
    order_fh   = fold.sort_values(["baseline_score_fh", "total_impressions_full"], ascending=[False, False]).index
    order_model = fold["oof_rf_score_calibrated"].sort_values(ascending=False).index

    for k in KS:
        records.append({
            "fold": fold_num, "k": k,
            "baseline_full_month": precision_at_k(fold["is_declining_proxy"].loc[order_full].values, k),
            "baseline_fh_flag_only": precision_at_k(fold["is_declining_proxy"].loc[order_fh].values, k),
            "model_calibrated": precision_at_k(fold["is_declining_proxy"].loc[order_model].values, k),
        })

df = pd.DataFrame(records)

print("=== Fold-by-fold at K=50 (tiebreak held constant) ===")
print(df[df["k"] == 50].to_string(index=False))
print()
print("=== Mean across folds, all K ===")
print(df.groupby("k")[["baseline_full_month", "baseline_fh_flag_only", "model_calibrated"]].mean().round(3))

=== Fold-by-fold at K=50 (tiebreak held constant) ===
 fold  k  baseline_full_month  baseline_fh_flag_only  model_calibrated
    1 50                 0.48                   0.42              0.66
    2 50                 0.54                   0.38              0.58
    3 50                 0.28                   0.24              0.50
    4 50                 0.46                   0.34              0.52
    5 50                 0.34                   0.32              0.52

=== Mean across folds, all K ===
     baseline_full_month  baseline_fh_flag_only  model_calibrated
k                                                                
20                 0.490                  0.340             0.530
50                 0.420                  0.340             0.556
100                0.412                  0.348             0.580
200                0.433                  0.359             0.577


## 2. Staleness & shape sanity

Before trusting anything downstream check if is this actually the data it claims to be, and whether the freshly-built population match the freshly-built queue?


In [ ]:
check(
    "model_df has no duplicate content_hash_id rows",
    model_df["content_hash_id"].is_unique,
    detail=f"{model_df['content_hash_id'].duplicated().sum()} duplicate id(s)",
)

check(
    "queue_df has no duplicate content_hash_id rows",
    queue_df["content_hash_id"].is_unique,
    detail=f"{queue_df['content_hash_id'].duplicated().sum()} duplicate id(s)",
)

check(
    "queue_df is the same population as model_df (same content_hash_id set)",
    set(model_df["content_hash_id"]) == set(queue_df["content_hash_id"]),
    detail=f"model_df={len(model_df):,} rows, queue_df={len(queue_df):,} rows",
)

check(
    "no missing client_hash_id",
    model_df["client_hash_id"].notna().all(),
    detail=f"{model_df['client_hash_id'].isna().sum()} missing",
)

for col in ["oof_rf_score", "oof_rf_score_calibrated", "archetype", "coverage", "is_declining_proxy"]:
    check(f"no NaNs in {col}", queue_df[col].notna().all() if col in queue_df.columns else model_df[col].notna().all())

pop_size = len(model_df)
check(
    "population size is on the order of hundreds of thousands (not the 'millions of pages' error caught earlier)",
    100_000 <= pop_size <= 999_999,
    detail=f"population_size={pop_size:,}",
    severity="warn",
)


✅ model_df has no duplicate content_hash_id rows — 0 duplicate id(s)
✅ queue_df has no duplicate content_hash_id rows — 0 duplicate id(s)
✅ queue_df is the same population as model_df (same content_hash_id set) — model_df=150,675 rows, queue_df=150,675 rows
✅ no missing client_hash_id — 0 missing
✅ no NaNs in oof_rf_score
✅ no NaNs in oof_rf_score_calibrated
✅ no NaNs in archetype
✅ no NaNs in coverage
✅ no NaNs in is_declining_proxy
✅ population size is on the order of hundreds of thousands (not the 'millions of pages' error caught earlier) — population_size=150,675


True

## 3. Label lineage — is `is_declining_proxy` actually what it claims to be?

Independently recomputed straight from the raw warehouse rows in pandas. If this drifts, the SQL and the documented definition ("impressions second half < impressions first half, first half > 0") have come apart.


In [ ]:
print("Querying raw daily rows for an independent label recompute (this can take a minute)...")

con = duckdb.connect()
hf_token = os.environ["HF_TOKEN"]
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

raw = con.sql(f"""
    SELECT content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{HF_MONTH_PATH}')
    WHERE gsc_data_available IS TRUE
""").df()
print(f"raw: {raw.shape}")

fh_mask = raw["report_date"] < HALF_SPLIT_DATE
sh_mask = raw["report_date"] >= HALF_SPLIT_DATE

impr_fh = raw[fh_mask].groupby("content_hash_id")["gsc_impressions"].sum()
impr_sh = raw[sh_mask].groupby("content_hash_id")["gsc_impressions"].sum()

label_base = impr_fh[impr_fh > 0]
label_indep = (impr_sh.reindex(label_base.index).fillna(0) < label_base).astype(int)
label_indep.name = "is_declining_proxy_indep"

Querying raw daily rows for an independent label recompute (this can take a minute)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw: (3611061, 5)


In [ ]:
compare = model_df.set_index("content_hash_id")[["is_declining_proxy"]].join(label_indep, how="inner")
mismatches = compare[compare["is_declining_proxy"] != compare["is_declining_proxy_indep"]]

check(
    "is_declining_proxy matches an independent pandas recompute, row for row",
    len(mismatches) == 0,
    detail=f"{len(mismatches):,} mismatches out of {len(compare):,} rows compared",
)

check(
    "label base rate is not degenerate (not ~0% or ~100%, which would mean a broken comparison)",
    0.05 < model_df["is_declining_proxy"].mean() < 0.95,
    detail=f"base rate = {model_df['is_declining_proxy'].mean():.3f}",
)


✅ is_declining_proxy matches an independent pandas recompute, row for row — 0 mismatches out of 150,675 rows compared
✅ label base rate is not degenerate (not ~0% or ~100%, which would mean a broken comparison) — base rate = 0.438


np.True_

## 4. Feature leakage — first-half-only discipline

Two regression tests tied directly to documented history: `total_impressions` and the `eligible` flag leaked the label once already. These assert that mistake can't silently come back, plus an independent recompute of the first-half features themselves.


In [ ]:
check(
    "FEATURE_COLS does not include total_impressions_full or eligible (the leak found earlier)",
    not any(c in FEATURE_COLS for c in ["total_impressions_full", "total_impressions", "eligible"]),
    detail=f"FEATURE_COLS={FEATURE_COLS}",
)

# 'eligible' and 'total_impressions_full' are expected to exist as columns in model_df —
# eligible gates the position-trend merge, total_impressions_full backs the coverage tier.
check(
    "'total_impressions' (bare) is not present — only the qualified _fh/_full variants",
    "total_impressions" not in model_df.columns,
    detail="'total_impressions' would be ambiguous with total_impressions_fh/_full and was the literal name of the original leak",
)

✅ FEATURE_COLS does not include total_impressions_full or eligible (the leak found earlier) — FEATURE_COLS=['avg_position_fh', 'log_impressions_fh', 'log_clicks_fh', 'ctr_fh', 'position_change', 'has_position_trend']
✅ 'total_impressions' (bare) is not present — only the qualified _fh/_full variants — 'total_impressions' would be ambiguous with total_impressions_fh/_full and was the literal name of the original leak


True

In [ ]:
fh = raw[fh_mask].copy()
impr_fh_sum = fh.groupby("content_hash_id")["gsc_impressions"].sum()
clicks_fh_sum = fh.groupby("content_hash_id")["gsc_clicks"].sum()
pos_fh_mean = fh[fh["gsc_avg_position"] > 0].groupby("content_hash_id")["gsc_avg_position"].mean()

feat_indep = pd.DataFrame({
    "log_impressions_fh_indep": np.log1p(impr_fh_sum),
    "log_clicks_fh_indep": np.log1p(clicks_fh_sum),
    "avg_position_fh_indep": pos_fh_mean,
})
feat_indep["ctr_fh_indep"] = clicks_fh_sum / impr_fh_sum

feat_compare = model_df.set_index("content_hash_id")[
    ["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh"]
].join(feat_indep, how="inner")

for col in ["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh"]:
    diffs = (feat_compare[col] - feat_compare[f"{col}_indep"]).abs()
    check(
        f"{col} matches independent first-half-only recompute",
        (diffs < 1e-6).all(),
        detail=f"max abs diff = {diffs.max():.6g}",
    )


✅ avg_position_fh matches independent first-half-only recompute — max abs diff = 4.26326e-14
✅ log_impressions_fh matches independent first-half-only recompute — max abs diff = 1.77636e-15
✅ log_clicks_fh matches independent first-half-only recompute — max abs diff = 8.88178e-16
✅ ctr_fh matches independent first-half-only recompute — max abs diff = 9.97466e-17


In [ ]:
# Sanity that the half-split is actually being applied — if someone accidentally reverted to full-month aggregation,
# avg_position_fh would equal the full-month average instead of the first-half-only average. They should generally differ.
full_pos_mean = raw[raw["gsc_avg_position"] > 0].groupby("content_hash_id")["gsc_avg_position"].mean()
full_vs_fh = model_df.set_index("content_hash_id")["avg_position_fh"].to_frame().join(
    full_pos_mean.rename("avg_position_full_check"), how="inner"
)
share_identical = (full_vs_fh["avg_position_fh"].round(6) == full_vs_fh["avg_position_full_check"].round(6)).mean()
check(
    "avg_position_fh is NOT just the full-month average in disguise",
    share_identical < 0.5,
    detail=f"{share_identical:.1%} of rows have fh == full-month position (expect low, not ~100%)",
)


✅ avg_position_fh is NOT just the full-month average in disguise — 7.5% of rows have fh == full-month position (expect low, not ~100%)


np.True_

## 5. Split integrity — grouped by client, no group split across folds

In [ ]:
per_client_folds = model_df.groupby("client_hash_id")["fold_id"].nunique()
check(
    "every client_hash_id appears in exactly one fold (GroupKFold is doing its job)",
    (per_client_folds == 1).all(),
    detail=f"{(per_client_folds != 1).sum()} client(s) split across folds",
)

check(
    "every row got a fold_id (none left unscored)",
    (model_df["fold_id"] > 0).all(),
    detail=f"{(model_df['fold_id'] <= 0).sum()} rows without a valid fold_id",
)

fold_share = model_df["fold_id"].value_counts(normalize=True)
check(
    "fold sizes are roughly balanced (~20% each)",
    ((fold_share - 0.2).abs() < 0.03).all(),
    detail=fold_share.round(3).to_dict(),
)


✅ every client_hash_id appears in exactly one fold (GroupKFold is doing its job) — 0 client(s) split across folds
✅ every row got a fold_id (none left unscored) — 0 rows without a valid fold_id
✅ fold sizes are roughly balanced (~20% each) — {3: 0.201, 1: 0.2, 5: 0.2, 4: 0.2, 2: 0.2}


np.True_

## 6. Calibration — within-fold percentile rank, exactly as documented

Percentile rank can't invert order within a fold. If calibrated and raw scores disagree
on ranking within the same fold, the calibration function itself has a bug.


In [ ]:
from scipy.stats import spearmanr

rank_breaks = 0
for fold in sorted(model_df["fold_id"].unique()):
    sub = model_df[model_df["fold_id"] == fold]
    rho, _ = spearmanr(sub["oof_rf_score"], sub["oof_rf_score_calibrated"])
    if rho < 0.999:
        rank_breaks += 1

check(
    "oof_rf_score_calibrated preserves within-fold rank order of oof_rf_score exactly",
    rank_breaks == 0,
    detail=f"{rank_breaks} fold(s) with Spearman rho < 0.999 between raw and calibrated",
)

cal_bounds = model_df.groupby("fold_id")["oof_rf_score_calibrated"].agg(["min", "max"])
check(
    "calibrated score spans ~0 to 1 within every fold",
    ((cal_bounds["min"] < 0.05) & (cal_bounds["max"] > 0.95)).all(),
    detail=cal_bounds.round(3).to_dict(orient="index"),
)


✅ oof_rf_score_calibrated preserves within-fold rank order of oof_rf_score exactly — 0 fold(s) with Spearman rho < 0.999 between raw and calibrated
✅ calibrated score spans ~0 to 1 within every fold — {1: {'min': 0.0, 'max': 1.0}, 2: {'min': 0.0, 'max': 1.0}, 3: {'min': 0.0, 'max': 1.0}, 4: {'min': 0.0, 'max': 1.0}, 5: {'min': 0.0, 'max': 1.0}}


np.True_

In [ ]:
# Top-K fold representation should be close to 20% per fold at every K — the whole reason calibration exists.
for k in [20, 50, 100, 200]:
    top_k = model_df.sort_values("oof_rf_score_calibrated", ascending=False).head(k)
    share = top_k["fold_id"].value_counts(normalize=True)
    max_share = share.max()
    check(
        f"K={k}: no single fold dominates the top-K queue (calibrated score)",
        max_share <= 0.30,
        detail=f"most-represented fold at {max_share:.0%} (expect ~20%)",
    )


✅ K=20: no single fold dominates the top-K queue (calibrated score) — most-represented fold at 20% (expect ~20%)
✅ K=50: no single fold dominates the top-K queue (calibrated score) — most-represented fold at 20% (expect ~20%)
✅ K=100: no single fold dominates the top-K queue (calibrated score) — most-represented fold at 20% (expect ~20%)
✅ K=200: no single fold dominates the top-K queue (calibrated score) — most-represented fold at 20% (expect ~20%)


## 7. Archetype logic — independently re-derived from the written spec

This reimplements `assign_archetype` from scratch directly from the documented branching order in `w07_pipeline_utils.py`'s docstring. If this disagrees with the live `archetype` column anywhere, either this reimplementation is wrong or the shipped function drifted from its own documented spec.

In [ ]:
high_score_cut = model_df.loc[model_df["baseline_score"] == 0, "oof_rf_score_calibrated"].quantile(HIGH_SCORE_PERCENTILE)
large_swing_cut = model_df["position_change"].abs().quantile(LARGE_SWING_PERCENTILE)
low_data_cut = model_df["total_impressions_full"].quantile(LOW_DATA_PERCENTILE)
boundary_margin = BOUNDARY_MARGIN_PCT * high_score_cut

zc = model_df["zero_clicks_at_position"] == 1
pw = model_df["position_worsened"] == 1
model_only_mask = (
    (~zc) & (~pw)
    & (model_df["baseline_score"] == 0)
    & (model_df["oof_rf_score_calibrated"] >= high_score_cut)
)

archetype_indep_arr = np.select(
    [zc & pw, zc & ~pw, (~zc) & pw, model_only_mask],
    ["zero_clicks_and_worsened", "zero_clicks_only", "position_worsened_only", "model_only_catch"],
    default="no_flag",
)
# Keep it aligned to model_df's own row order for reuse below, and as an id-indexed Series for comparing against queue_df, which is sorted by score — a different row order.
archetype_indep = pd.Series(archetype_indep_arr, index=model_df["content_hash_id"], name="archetype_indep")

compare = queue_df.set_index("content_hash_id")[["archetype"]].join(archetype_indep, how="inner")
archetype_mismatches = (compare["archetype"] != compare["archetype_indep"]).sum()

check(
    "archetype matches an independently re-derived assignment, row for row",
    archetype_mismatches == 0,
    detail=f"{archetype_mismatches:,} mismatches out of {len(compare):,} rows",
)

check(
    "every row gets exactly one of the 5 known archetypes (no unexpected label)",
    set(queue_df["archetype"].unique()) <= {
        "zero_clicks_and_worsened", "zero_clicks_only", "position_worsened_only",
        "model_only_catch", "no_flag",
    },
    detail=f"observed: {sorted(queue_df['archetype'].unique())}",
)

✅ archetype matches an independently re-derived assignment, row for row — 0 mismatches out of 150,675 rows
✅ every row gets exactly one of the 5 known archetypes (no unexpected label) — observed: ['model_only_catch', 'no_flag', 'position_worsened_only', 'zero_clicks_and_worsened', 'zero_clicks_only']


True

In [ ]:
low_data_mask = model_df["total_impressions_full"] < low_data_cut
near_boundary_mask = (model_df["oof_rf_score_calibrated"] - high_score_cut).abs() < boundary_margin

coverage_indep_arr = np.where(
    low_data_mask, "low",
    np.where(model_only_mask & near_boundary_mask, "low", "high"),
)
coverage_indep = pd.Series(coverage_indep_arr, index=model_df["content_hash_id"], name="coverage_indep")

compare_cov = queue_df.set_index("content_hash_id")[["coverage"]].join(coverage_indep, how="inner")
coverage_mismatches = (compare_cov["coverage"] != compare_cov["coverage_indep"]).sum()

check(
    "coverage matches an independently re-derived assignment, row for row",
    coverage_mismatches == 0,
    detail=f"{coverage_mismatches:,} mismatches out of {len(compare_cov):,} rows",
)

✅ coverage matches an independently re-derived assignment, row for row — 0 mismatches out of 150,675 rows


np.True_

## 8. Coverage naming honesty — regression test for the reason it was renamed

`coverage` was renamed from `confidence` because low-tier rows turned out to have a higher real decline rate than high-tier rows in every archetype — i.e. it never tracked reliability. This checks that finding still holds.


In [ ]:
outcome_by_cov = queue_df.groupby(["archetype", "coverage"])["is_declining_proxy"].mean().unstack()
print(outcome_by_cov.round(3))

if {"low", "high"}.issubset(outcome_by_cov.columns):
    still_inverted = (outcome_by_cov["low"] >= outcome_by_cov["high"]).all()
    check(
        "low-coverage rows still show >= decline rate than high-coverage rows in every archetype",
        bool(still_inverted),
        detail="matches the documented finding behind the confidence→coverage rename" if still_inverted
               else "the inversion no longer holds in every archetype — recheck the Limitations claim",
        severity="warn",
    )


coverage                   high    low
archetype                             
model_only_catch          0.449  0.536
no_flag                   0.331  0.579
position_worsened_only    0.436  0.469
zero_clicks_and_worsened  0.529  0.606
zero_clicks_only          0.421  0.472
✅ low-coverage rows still show >= decline rate than high-coverage rows in every archetype — matches the documented finding behind the confidence→coverage rename


## 9. Baseline rule reproduction

In [ ]:
baseline_indep = model_df["zero_clicks_at_position"] * 2 + model_df["position_worsened"]
check(
    "baseline_score matches zero_clicks_at_position*2 + position_worsened exactly",
    (model_df["baseline_score"] == baseline_indep).all(),
    detail=f"{(model_df['baseline_score'] != baseline_indep).sum()} mismatches",
)


✅ baseline_score matches zero_clicks_at_position*2 + position_worsened exactly — 0 mismatches


np.True_

## 10. Precision@K — recomputed with an independent implementation

Same GroupKFold reconstruction 03 uses, but the ranking and precision@K math below is written fresh, not imported from `03_validate_deployed_score.py`.


In [ ]:
from sklearn.model_selection import GroupKFold

def precision_at_k(labels_sorted_by_score_desc, k):
    return float(np.asarray(labels_sorted_by_score_desc)[:k].mean())

X = model_df[FEATURE_COLS].astype(float)
y = model_df["is_declining_proxy"].astype(int)
groups = model_df["client_hash_id"]
gkf = GroupKFold(n_splits=N_FOLDS)

KS = [20, 50, 100, 200]
records = []
for fold_num, (_, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
    fold_rows = model_df.iloc[test_idx]
    order = fold_rows["oof_rf_score_calibrated"].sort_values(ascending=False).index
    ranked_labels = y.loc[order].values
    for k in KS:
        records.append({"fold": fold_num, "k": k, "precision_indep": precision_at_k(ranked_labels, k)})

precision_indep = pd.DataFrame(records).groupby("k")["precision_indep"].mean().round(3)
print(precision_indep)


k
20     0.530
50     0.556
100    0.580
200    0.577
Name: precision_indep, dtype: float64


In [ ]:
live_precision = load_live("capstone_precision_at_k.json")
committed_precision = load_committed("capstone_precision_at_k.json")

for k in KS:
    live_mean = live_precision["summary_mean_std"][str(k)]["oof_rf_score_calibrated"]["mean"]
    check(
        f"K={k}: independent precision@K recompute matches 03's own fresh output",
        close(precision_indep[k], live_mean, tol=0.01),
        detail=f"independent={precision_indep[k]:.3f}, script={live_mean:.3f}",
    )

for k in KS:
    committed_mean = committed_precision["summary_mean_std"][str(k)]["oof_rf_score_calibrated"]["mean"]
    live_mean = live_precision["summary_mean_std"][str(k)]["oof_rf_score_calibrated"]["mean"]
    check(
        f"K={k}: fresh precision@K hasn't drifted far from the committed (locked) result",
        close(committed_mean, live_mean, tol=0.03),
        detail=f"committed={committed_mean:.3f}, fresh={live_mean:.3f} "
               "(small drift expected from DuckDB aggregation-order nondeterminism)",
        severity="warn",
    )


✅ K=20: independent precision@K recompute matches 03's own fresh output — independent=0.530, script=0.530
✅ K=50: independent precision@K recompute matches 03's own fresh output — independent=0.556, script=0.556
✅ K=100: independent precision@K recompute matches 03's own fresh output — independent=0.580, script=0.580
✅ K=200: independent precision@K recompute matches 03's own fresh output — independent=0.577, script=0.577
✅ K=20: fresh precision@K hasn't drifted far from the committed (locked) result — committed=0.530, fresh=0.530 (small drift expected from DuckDB aggregation-order nondeterminism)
✅ K=50: fresh precision@K hasn't drifted far from the committed (locked) result — committed=0.556, fresh=0.556 (small drift expected from DuckDB aggregation-order nondeterminism)
✅ K=100: fresh precision@K hasn't drifted far from the committed (locked) result — committed=0.580, fresh=0.580 (small drift expected from DuckDB aggregation-order nondeterminism)
✅ K=200: fresh precision@K hasn't dr

## 11. Cross-file consistency — committed vs. freshly regenerated

Catches the failure mode that prompted this notebook: a JSON file sitting in the repo that no longer matches what the pipeline actually produces. Small differences are expected due to documented DuckDB nondeterminism; large ones mean something is genuinely stale or the pipeline changed without the committed outputs being refreshed.


In [ ]:
committed_metrics = load_committed("w07_metrics.json")
live_metrics = load_live("w07_metrics.json")

check(
    "population_size: committed vs. fresh",
    close(committed_metrics["population_size"], live_metrics["population_size"], tol=live_metrics["population_size"] * 0.02),
    detail=f"committed={committed_metrics['population_size']:,}, fresh={live_metrics['population_size']:,}",
    severity="warn",
)

for archetype in set(committed_metrics["archetype_counts"]) | set(live_metrics["archetype_counts"]):
    c = committed_metrics["archetype_counts"].get(archetype, 0)
    l = live_metrics["archetype_counts"].get(archetype, 0)
    check(
        f"archetype_counts[{archetype}]: committed vs. fresh",
        close(c, l, tol=max(50, l * 0.03)),
        detail=f"committed={c:,}, fresh={l:,}",
        severity="warn",
    )

for tier in ["high", "low"]:
    c = committed_metrics["coverage_split"].get(tier, 0)
    l = live_metrics["coverage_split"].get(tier, 0)
    check(
        f"coverage_split[{tier}]: committed vs. fresh",
        close(c, l, tol=max(50, l * 0.03)),
        detail=f"committed={c:,}, fresh={l:,}",
        severity="warn",
    )


✅ population_size: committed vs. fresh — committed=150,675, fresh=150,675
✅ archetype_counts[model_only_catch]: committed vs. fresh — committed=7,286, fresh=7,286
✅ archetype_counts[zero_clicks_and_worsened]: committed vs. fresh — committed=10,609, fresh=10,609
✅ archetype_counts[zero_clicks_only]: committed vs. fresh — committed=14,457, fresh=14,457
✅ archetype_counts[no_flag]: committed vs. fresh — committed=65,571, fresh=65,570
✅ archetype_counts[position_worsened_only]: committed vs. fresh — committed=52,752, fresh=52,753
✅ coverage_split[high]: committed vs. fresh — committed=111,981, fresh=111,981
✅ coverage_split[low]: committed vs. fresh — committed=38,694, fresh=38,694


In [ ]:
committed_sentinel = load_committed("sentinel_fill_check.json")
live_sentinel = load_live("sentinel_fill_check.json")
check(
    "sentinel_fill_check.json: no_trend_share committed vs. fresh",
    close(committed_sentinel["no_trend_share"], live_sentinel["no_trend_share"], tol=0.02),
    detail=f"committed={committed_sentinel['no_trend_share']:.3f}, fresh={live_sentinel['no_trend_share']:.3f}",
    severity="warn",
)

# The ~21% sentinel-fill gap is a named limitation — confirm it's still roughly that instead of having silently grown into a much bigger share of the population.
check(
    "sentinel-fill gap (has_position_trend == 0) is still in the ballpark of the documented ~21%",
    0.10 <= live_sentinel["no_trend_share"] <= 0.35,
    detail=f"fresh no_trend_share={live_sentinel['no_trend_share']:.3f}",
    severity="warn",
)


✅ sentinel_fill_check.json: no_trend_share committed vs. fresh — committed=0.209, fresh=0.209
✅ sentinel-fill gap (has_position_trend == 0) is still in the ballpark of the documented ~21% — fresh no_trend_share=0.209


True

In [ ]:
# Chart + JSON files that should exist after a full run, and the stale file that should not.
expected_files = [
    "work/outputs/feature_importance.json",
    "work/outputs/sentinel_fill_check.json",
    "work/outputs/queue_diagnostics.json",
    "work/outputs/fold_representation_check.json",
    "work/outputs/capstone_precision_at_k.json",
    "work/outputs/w07_metrics.json",
    "work/outputs/w07_report.md",
    "work/outputs/charts/archetype_mix.svg",
    "work/outputs/charts/coverage_mix.svg",
    "work/outputs/charts/feature_importance.svg",
    "work/outputs/charts/precision_at_k.svg",
    "work/outputs/charts/score_distribution_by_archetype.svg",
]
missing = [f for f in expected_files if not os.path.isfile(f)]
check("all expected output files exist after a full run", len(missing) == 0, detail=f"missing: {missing}")

check(
    "stale charts/confidence_mix.svg has not reappeared",
    not os.path.isfile("work/outputs/charts/confidence_mix.svg"),
)


✅ all expected output files exist after a full run — missing: []
✅ stale charts/confidence_mix.svg has not reappeared


True

## 12. Report cross-check

`work/capstone_report.md` isn't written yet as of this notebook's creation, so this section is a placeholder that will WARN-and-skip until there's real prose to check numbers against. Once it's written, replace the `TODO` check below with real regex pulls of any hardcoded numbers (population size, precision@K, archetype counts) and compare them against `w07_metrics.json` / `capstone_precision_at_k.json` to make sure the paper doesn't quote a number that no longer matches the JSON.


In [ ]:
report_path = "work/capstone_report.md"
template_path = "work/capstone_report_template.md"

if os.path.isfile(report_path) and os.path.isfile(template_path):
    with open(report_path) as f:
        report_text = f.read()
    with open(template_path) as f:
        template_text = f.read()

    if report_text.strip() == template_text.strip():
        check(
            "capstone_report.md has been filled in (not just the template)",
            False,
            detail="still byte-identical to capstone_report_template.md — numeric cross-check skipped",
            severity="warn",
        )
    else:
        # TODO once the report is written: pull hardcoded numbers out with regex and compare against w07_metrics.json / capstone_precision_at_k.json here.
        check(
            "capstone_report.md differs from the template (something has been written)",
            True,
            detail="numeric cross-check against JSON not yet implemented — see the TODO above",
            severity="warn",
        )
else:
    check("capstone_report.md / template found on disk", False, severity="warn")


⚠️ capstone_report.md has been filled in (not just the template) — still byte-identical to capstone_report_template.md — numeric cross-check skipped


## 13. Summary

In [ ]:
summary = pd.DataFrame(RESULTS)
n_fail = (summary["status"] == "FAIL").sum()
n_warn = (summary["status"] == "WARN").sum()
n_pass = (summary["status"] == "PASS").sum()

print(summary.to_string(index=False))
print(f"\n{len(summary)} checks — {n_pass} PASS, {n_warn} WARN, {n_fail} FAIL")

if n_fail:
    raise AssertionError(
        f"{n_fail} check(s) FAILED — see the table above. Do not trust the capstone "
        "notebook's numbers or the paper until these are resolved."
    )
print("\nNo hard failures. WARNs above are worth a skim before finalizing the paper, "
      "but nothing here blocks moving forward.")


                                                                                                       test status                                                                                                                                            detail
                                                             model_df has no duplicate content_hash_id rows   PASS                                                                                                                                 0 duplicate id(s)
                                                             queue_df has no duplicate content_hash_id rows   PASS                                                                                                                                 0 duplicate id(s)
                                     queue_df is the same population as model_df (same content_hash_id set)   PASS                                                                                                      m